# Análisis de representaciones de reseñas: BoW, TF-IDF y Word2Vec


In [5]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display


def localizar_proyecto():
    actual = Path.cwd().resolve()
    candidatos = [actual, *actual.parents]
    candidatos += [h for p in candidatos[:2] for h in p.iterdir() if h.is_dir() and 'pos-tagging' in h.name.lower()]
    for carpeta in candidatos:
        if all((carpeta / d).is_dir() for d in ('data', 'src', 'notebooks')):
            return carpeta
    raise FileNotFoundError(
        'No se pudo identificar la estructura del repositorio (data, src y notebooks).'
    )

RAIZ = localizar_proyecto()
DATOS = RAIZ / 'data'
SALIDA = RAIZ / 'outputs' / 'persona2'
SALIDA.mkdir(parents=True, exist_ok=True)
if str(RAIZ / 'src') not in sys.path:
    sys.path.insert(0, str(RAIZ / 'src'))
print('Raíz del proyecto:', RAIZ)


Raíz del proyecto: C:\Users\marie\pos-tagging-resenas-cine-series


In [6]:
from pathlib import Path
import pandas as pd

DATOS = RAIZ / "data"
SALIDA = RAIZ / "outputs" / "persona2"
SALIDA.mkdir(parents=True, exist_ok=True)

print("ARCHIVOS DISPONIBLES EN DATA:\n")

for archivo in sorted(DATOS.rglob("*")):
    if archivo.is_file():
        print(archivo.relative_to(RAIZ))

print("\nBUSCANDO ARCHIVOS NECESARIOS:\n")

candidatos_pos = [
    DATOS / "results" / "pos_analysis.csv",
    DATOS / "results" / "pos_analysis.csv.gz"
]

ruta_pos = next(
    (p for p in candidatos_pos if p.is_file()),
    None
)

ruta_resenas = DATOS / "processed" / "resenas_limpias.csv"

print("Archivo POS:", ruta_pos if ruta_pos else "No disponible")
print("Archivo reseñas:",
      ruta_resenas if ruta_resenas.is_file() else "No disponible")

if ruta_pos is not None:
    pos = pd.read_csv(ruta_pos)
    print("\nCOLUMNAS POS:")
    print(pos.columns.tolist())
    display(pos.head())

if ruta_resenas.is_file():
    resenas = pd.read_csv(ruta_resenas)
    print("\nCOLUMNAS RESEÑAS:")
    print(resenas.columns.tolist())
    display(resenas.head())


ARCHIVOS DISPONIBLES EN DATA:

data\processed\.gitignore
data\processed\esquema_mongodb.md
data\processed\resenas_es.csv
data\processed\resenas_limpias.csv
data\raw\.gitignore
data\results\.gitignore
data\results\pos_analysis.csv.gz

BUSCANDO ARCHIVOS NECESARIOS:

Archivo POS: C:\Users\marie\pos-tagging-resenas-cine-series\data\results\pos_analysis.csv.gz
Archivo reseñas: C:\Users\marie\pos-tagging-resenas-cine-series\data\processed\resenas_limpias.csv

COLUMNAS POS:
['id', 'oracion', 'token', 'lema', 'upos_spacy', 'morf', 'upos_nltk']


,id,oracion,token,lema,upos_spacy,morf,upos_nltk
0,1,0,Vi,ver,VERB,Mood=Ind|Number=Sing|Person=1|Tense=Past|VerbF...,VERB
1,1,0,esta,este,DET,Gender=Fem|Number=Sing|PronType=Dem,DET
2,1,0,película,película,NOUN,Gender=Fem|Number=Sing,NOUN
3,1,0,por,por,ADP,NaN,ADP
4,1,0,primera,primero,ADJ,Gender=Fem|NumType=Ord|Number=Sing,ADJ



COLUMNAS RESEÑAS:
['id', 'movie_id', 'genero', 'periodo', 'anio_review', 'fecha', 'calificacion', 'valoracion', 'is_spoiler', 'texto_en', 'texto_es']


,id,movie_id,genero,periodo,anio_review,fecha,calificacion,valoracion,is_spoiler,texto_en,texto_es
0,1,tt0092067,Animation,antes_2010,2002,2002-05-05,10,positiva,False,i watched this movie for the first time when i...,Vi esta película por primera vez cuando tenía ...
1,2,tt0116778,Comedy,2010_en_adelante,2012,2012-04-08,6,intermedia,False,Kingpin is a funny movie with some laugh out l...,Kingpin es una película divertida con algunas ...
2,3,tt0383534,Drama,2010_en_adelante,2016,2016-12-22,7,positiva,False,"In need of some Christmas spirit, I decided to...",En la necesidad de un poco de espíritu navideñ...
3,4,tt0295700,Horror,2010_en_adelante,2017,2017-10-18,9,positiva,False,"Wrong Turn has become a very big thing,from th...",Turno equivocado se ha convertido en una cosa ...
4,5,tt0320691,Action,2010_en_adelante,2016,2016-03-24,7,positiva,False,Sometimes I think that Underworld would have m...,A veces pienso que Inframundo habría hecho un ...


In [8]:
import re
import pandas as pd

pos_limpio = pos.copy()

pos_limpio = pos_limpio[
    pos_limpio["upos_spacy"].isin(
        ["NOUN", "VERB", "ADJ", "ADV"]
    )
].copy()

pos_limpio["palabra"] = (
    pos_limpio["token"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
)

pos_limpio = pos_limpio[
    pos_limpio["palabra"].str.fullmatch(
        r"[a-záéíóúüñ]{2,}",
        na=False
    )
].copy()

oraciones = (
    pos_limpio
    .groupby(["id", "oracion"], sort=False)["palabra"]
    .apply(list)
    .tolist()
)

oraciones = [
    oracion for oracion in oraciones
    if len(oracion) >= 2
]

documentos = (
    pos_limpio
    .groupby("id", sort=False)["palabra"]
    .apply(list)
    .reset_index(name="tokens")
)

documentos = documentos.sort_values(
    "id"
).reset_index(drop=True)

documentos["texto"] = documentos["tokens"].apply(
    lambda palabras: " ".join(palabras)
)

print("Preparación finalizada")
print("Reseñas procesadas:", len(documentos))
print("Oraciones para Word2Vec:", len(oraciones))
print("Tokens únicos:", pos_limpio["palabra"].nunique())

display(documentos.head())


Preparación finalizada
Reseñas procesadas: 6283
Oraciones para Word2Vec: 61360
Tokens únicos: 29862


,id,tokens,texto
0,1,"[vi, película, primera, vez, tenía, años, tien...",vi película primera vez tenía años tiene única...
1,2,"[película, divertida, escenas, risa, alto, his...",película divertida escenas risa alto historia ...
2,3,"[necesidad, espíritu, navideño, decidí, ver, p...",necesidad espíritu navideño decidí ver partir ...
3,4,"[turno, equivocado, convertido, cosa, muy, gra...",turno equivocado convertido cosa muy grande la...
4,5,"[veces, pienso, hecho, video, musical, mejor, ...",veces pienso hecho video musical mejor películ...


In [9]:
from sklearn.feature_extraction.text import (
    CountVectorizer,
    TfidfVectorizer
)
from scipy.sparse import save_npz
import pandas as pd

bow_vectorizador = CountVectorizer(
    min_df=1,
    lowercase=False
)

tfidf_vectorizador = TfidfVectorizer(
    min_df=1,
    lowercase=False
)

X_bow = bow_vectorizador.fit_transform(
    documentos["texto"]
)

X_tfidf = tfidf_vectorizador.fit_transform(
    documentos["texto"]
)

print("RESULTADOS BAG OF WORDS Y TF-IDF")
print("Matriz BoW:", X_bow.shape)
print("Matriz TF-IDF:", X_tfidf.shape)

frecuencias = X_bow.sum(axis=0).A1

tabla_bow = pd.DataFrame({
    "palabra": bow_vectorizador.get_feature_names_out(),
    "frecuencia": frecuencias
}).sort_values(
    "frecuencia", ascending=False
)

print("\nTOP 15 PALABRAS MÁS FRECUENTES")
display(tabla_bow.head(15))

promedios_tfidf = X_tfidf.mean(axis=0).A1

tabla_tfidf = pd.DataFrame({
    "palabra": tfidf_vectorizador.get_feature_names_out(),
    "tfidf_promedio": promedios_tfidf
}).sort_values(
    "tfidf_promedio", ascending=False
)

print("\nTOP 15 PALABRAS POR TF-IDF PROMEDIO")
display(tabla_tfidf.head(15))

save_npz(SALIDA / "bow.npz", X_bow)
save_npz(SALIDA / "tfidf.npz", X_tfidf)

documentos[["id"]].to_csv(
    SALIDA / "ids_resenas.csv",
    index=False
)

tabla_bow.to_csv(
    SALIDA / "frecuencias_bow.csv",
    index=False
)

tabla_tfidf.to_csv(
    SALIDA / "importancia_tfidf.csv",
    index=False
)

print("\nArchivos guardados")


RESULTADOS BAG OF WORDS Y TF-IDF
Matriz BoW: (6283, 29862)
Matriz TF-IDF: (6283, 29862)

TOP 15 PALABRAS MÁS FRECUENTES


,palabra,frecuencia
21122,película,21120
19760,no,15698
19441,más,6787
21124,películas,3984
19421,muy,3945
14692,historia,3755
29104,ver,3516
27452,tan,2966
27868,tiene,2948
23576,realmente,2829



TOP 15 PALABRAS POR TF-IDF PROMEDIO


,palabra,tfidf_promedio
21122,película,0.077549
19760,no,0.060102
19441,más,0.033071
19421,muy,0.025333
21124,películas,0.025284
14692,historia,0.024111
29104,ver,0.023424
27452,tan,0.020666
23576,realmente,0.020432
27868,tiene,0.019930



Archivos guardados


In [10]:
from gensim.models import Word2Vec

parametros = {
    "vector_size": 100,
    "window": 5,
    "min_count": 3,
    "workers": 1,
    "seed": 42,
    "epochs": 10
}

modelo_cbow = Word2Vec(
    sentences=oraciones,
    sg=0,
    **parametros
)

modelo_skipgram = Word2Vec(
    sentences=oraciones,
    sg=1,
    **parametros
)

modelo_cbow.save(
    str(SALIDA / "word2vec_cbow.model")
)

modelo_skipgram.save(
    str(SALIDA / "word2vec_skipgram.model")
)

print("Entrenamiento finalizado")
print("Vocabulario CBOW:", len(modelo_cbow.wv))
print("Vocabulario Skip-Gram:", len(modelo_skipgram.wv))
print("Dimensiones:", modelo_cbow.vector_size)

palabra = "película"

if palabra in modelo_cbow.wv:
    print("\nPALABRAS SIMILARES A 'PELÍCULA' - CBOW")
    print(modelo_cbow.wv.most_similar(palabra, topn=10))

if palabra in modelo_skipgram.wv:
    print("\nPALABRAS SIMILARES A 'PELÍCULA' - SKIP-GRAM")
    print(modelo_skipgram.wv.most_similar(palabra, topn=10))


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Entrenamiento finalizado
Vocabulario CBOW: 13232
Vocabulario Skip-Gram: 13232
Dimensiones: 100

PALABRAS SIMILARES A 'PELÍCULA' - CBOW
[('experiencia', 0.6890494227409363), ('característica', 0.6593263745307922), ('secuela', 0.6492540836334229), ('comedia', 0.6352521181106567), ('sorpresa', 0.6049805283546448), ('embargo', 0.6033974289894104), ('diría', 0.5952327251434326), ('entretenida', 0.5932117700576782), ('visualización', 0.5885298848152161), ('caricatura', 0.5795506834983826)]

PALABRAS SIMILARES A 'PELÍCULA' - SKIP-GRAM
[('timón', 0.7088034749031067), ('molestaron', 0.7070832848548889), ('facturado', 0.7065701484680176), ('evolución', 0.7006244659423828), ('erótica', 0.7004095911979675), ('calificaría', 0.6995341181755066), ('divertí', 0.6978106498718262), ('inmensamente', 0.6958972215652466), ('decentemente', 0.6955767273902893), ('estaré', 0.6929649114608765)]


In [11]:
print("VERIFICACIÓN DE MODELOS")

for nombre in ["modelo_cbow", "modelo_skipgram"]:
    modelo = globals().get(nombre)

    if modelo is None:
        print(nombre, ": No disponible")
    else:
        print(nombre, ": Disponible")
        print("Vocabulario:", len(modelo.wv))
        print("Dimensiones:", modelo.vector_size)

print("\nARCHIVOS GUARDADOS:")

for archivo in [
    "word2vec_cbow.model",
    "word2vec_skipgram.model"
]:
    ruta = SALIDA / archivo
    print(archivo, ":", ruta.is_file())


VERIFICACIÓN DE MODELOS
modelo_cbow : Disponible
Vocabulario: 13232
Dimensiones: 100
modelo_skipgram : Disponible
Vocabulario: 13232
Dimensiones: 100

ARCHIVOS GUARDADOS:
word2vec_cbow.model : True
word2vec_skipgram.model : True


In [12]:
import pandas as pd
import unicodedata

def quitar_tildes(texto):
    texto = unicodedata.normalize("NFD", texto.lower())
    return "".join(
        letra for letra in texto
        if unicodedata.category(letra) != "Mn"
    )

def buscar_palabra(modelo, palabra):
    palabra = palabra.lower()

    if palabra in modelo.wv:
        return palabra

    equivalente = quitar_tildes(palabra)

    for termino in modelo.wv.index_to_key:
        if quitar_tildes(termino) == equivalente:
            return termino

    return None

campos_semanticos = {
    "Cine": ["película", "actor", "director", "guion"],
    "Emociones": ["miedo", "alegría", "tristeza"],
    "Géneros": ["terror", "comedia", "drama", "acción"],
    "Valoraciones": ["excelente", "malo", "aburrido"]
}

resultados_semanticos = []

for nombre, modelo in [
    ("CBOW", modelo_cbow),
    ("Skip-Gram", modelo_skipgram)
]:
    for campo, palabras in campos_semanticos.items():
        for palabra in palabras:

            encontrada = buscar_palabra(modelo, palabra)

            if encontrada is None:
                resultados_semanticos.append({
                    "modelo": nombre,
                    "campo": campo,
                    "palabra": palabra,
                    "termino_usado": None,
                    "similar": "Fuera del vocabulario",
                    "similitud": None
                })
                continue

            similares = modelo.wv.most_similar(
                encontrada,
                topn=5
            )

            for termino, puntuacion in similares:
                resultados_semanticos.append({
                    "modelo": nombre,
                    "campo": campo,
                    "palabra": palabra,
                    "termino_usado": encontrada,
                    "similar": termino,
                    "similitud": round(float(puntuacion), 4)
                })

tabla_semantica = pd.DataFrame(resultados_semanticos)

print("RESULTADOS DE CAMPOS SEMÁNTICOS")
display(tabla_semantica)

tabla_semantica.to_csv(
    SALIDA / "campos_semanticos.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nArchivo guardado: campos_semanticos.csv")


RESULTADOS DE CAMPOS SEMÁNTICOS


,modelo,campo,palabra,termino_usado,similar,similitud
0,CBOW,Cine,película,película,experiencia,0.6890
1,CBOW,Cine,película,película,característica,0.6593
2,CBOW,Cine,película,película,secuela,0.6493
3,CBOW,Cine,película,película,comedia,0.6353
4,CBOW,Cine,película,película,sorpresa,0.6050
...,...,...,...,...,...,...
135,Skip-Gram,Valoraciones,aburrido,aburrido,apresurado,0.7657
136,Skip-Gram,Valoraciones,aburrido,aburrido,repetitivo,0.7626
137,Skip-Gram,Valoraciones,aburrido,aburrido,pretencioso,0.7511
138,Skip-Gram,Valoraciones,aburrido,aburrido,tedioso,0.7443



Archivo guardado: campos_semanticos.csv


In [13]:
import pandas as pd


analogias = [
    ("actor", "hombre", "mujer", "actriz"),
    ("director", "hombre", "mujer", "directora"),
    ("rey", "hombre", "mujer", "reina"),
    ("bueno", "malo", "triste", "alegre"),
    ("terror", "miedo", "risa", "comedia"),
    ("comedia", "risa", "miedo", "terror")
]

resultados_analogias = []

for nombre, modelo in [
    ("CBOW", modelo_cbow),
    ("Skip-Gram", modelo_skipgram)
]:
    for numero, (a, b, c, esperado) in enumerate(
        analogias, start=1
    ):

        palabra_a = buscar_palabra(modelo, a)
        palabra_b = buscar_palabra(modelo, b)
        palabra_c = buscar_palabra(modelo, c)
        palabra_esperada = buscar_palabra(modelo, esperado)

        if None in [palabra_a, palabra_b, palabra_c]:
            predicciones = []
            estado = "No evaluable: vocabulario insuficiente"
        else:
            predicciones = modelo.wv.most_similar(
                positive=[palabra_a, palabra_c],
                negative=[palabra_b],
                topn=5
            )
            estado = "Evaluada"

        palabras_predichas = [
            palabra for palabra, similitud in predicciones
        ]

        acierto = (
            palabra_esperada is not None
            and palabra_esperada in palabras_predichas
        )

        resultados_analogias.append({
            "modelo": nombre,
            "analogia": numero,
            "operacion": f"{a} - {b} + {c}",
            "esperado": esperado,
            "predicciones_top5": ", ".join(
                palabras_predichas
            ),
            "acierto_top5": acierto,
            "estado": estado
        })

tabla_analogias = pd.DataFrame(resultados_analogias)

print("RESULTADOS DE LAS SEIS ANALOGÍAS")
display(tabla_analogias)

print("\nRESUMEN POR MODELO")
display(
    tabla_analogias.groupby("modelo")[
        "acierto_top5"
    ].sum().reset_index(name="aciertos")
)

tabla_analogias.to_csv(
    SALIDA / "analogias_word2vec.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nArchivo guardado: analogias_word2vec.csv")


RESULTADOS DE LAS SEIS ANALOGÍAS


,modelo,analogia,operacion,esperado,predicciones_top5,acierto_top5,estado
0,CBOW,1,actor - hombre + mujer,actriz,"actriz, papel, casting, nominado, rendimiento",True,Evaluada
1,CBOW,2,director - hombre + mujer,directora,"producción, premios, nominado, ganó, cinematog...",False,Evaluada
2,CBOW,3,rey - hombre + mujer,reina,"moribunda, bañera, viuda, encubierto, solitario",False,Evaluada
3,CBOW,4,bueno - malo + triste,alegre,"encontré, hilarante, molesta, aterradora, dece...",False,Evaluada
4,CBOW,5,terror - miedo + risa,comedia,"slasher, citas, animada, altamente, anime",False,Evaluada
5,CBOW,6,comedia - risa + miedo,terror,"drama, fantasía, éxito, sentido, valor",False,Evaluada
6,Skip-Gram,1,actor - hombre + mujer,actriz,"actriz, interpretó, clava, interpretar, casting",True,Evaluada
7,Skip-Gram,2,director - hombre + mujer,directora,"productor, nominaciones, aseguró, actriz, aplauso",False,Evaluada
8,Skip-Gram,3,rey - hombre + mujer,reina,"médico, comandante, flor, apuros, casada",False,Evaluada
9,Skip-Gram,4,bueno - malo + triste,alegre,"desgarrador, reconfortante, inspirador, impact...",False,Evaluada



RESUMEN POR MODELO


,modelo,aciertos
0,CBOW,1
1,Skip-Gram,1



Archivo guardado: analogias_word2vec.csv


In [15]:
import numpy as np
import pandas as pd

def generar_vectores(modelo, documentos):
    vectores = []
    cantidad_tokens = []

    for tokens in documentos["tokens"]:
        palabras_validas = [
            palabra for palabra in tokens
            if palabra in modelo.wv
        ]

        cantidad_tokens.append(len(palabras_validas))

        if palabras_validas:
            vector = np.mean(
                modelo.wv[palabras_validas],
                axis=0
            )
        else:
            vector = np.zeros(
                modelo.vector_size,
                dtype=np.float32
            )

        vectores.append(vector)

    return np.asarray(vectores), cantidad_tokens


X_cbow, tokens_cbow = generar_vectores(
    modelo_cbow, documentos
)

X_skipgram, tokens_skipgram = generar_vectores(
    modelo_skipgram, documentos
)

assert len(documentos) == len(X_cbow)
assert len(documentos) == len(X_skipgram)

for nombre, matriz, cantidades in [
    ("cbow", X_cbow, tokens_cbow),
    ("skipgram", X_skipgram, tokens_skipgram)
]:
    tabla = pd.DataFrame(
        matriz,
        columns=[
            f"dim_{i}" for i in range(matriz.shape[1])
        ]
    )

    tabla.insert(
        0, "id", documentos["id"].to_numpy()
    )

    tabla["tokens_validos"] = cantidades

    tabla.to_csv(
        SALIDA / f"vectores_{nombre}.csv",
        index=False,
        encoding="utf-8-sig"
    )

print("Representaciones vectoriales generadas")

print("\nCBOW:")
print("Dimensiones:", X_cbow.shape)

print("\nSKIP-GRAM:")
print("Dimensiones:", X_skipgram.shape)

print("\nIDENTIFICADORES:")
print("Reseñas:", len(documentos))
print("IDs únicos:", documentos["id"].nunique())

print("\nARCHIVOS GUARDADOS:")
print("vectores_cbow.csv")
print("vectores_skipgram.csv")

display(
    pd.read_csv(
        SALIDA / "vectores_cbow.csv"
    ).iloc[:5, :8]
)


Representaciones vectoriales generadas

CBOW:
Dimensiones: (6283, 100)

SKIP-GRAM:
Dimensiones: (6283, 100)

IDENTIFICADORES:
Reseñas: 6283
IDs únicos: 6283

ARCHIVOS GUARDADOS:
vectores_cbow.csv
vectores_skipgram.csv


,id,dim_0,dim_1,dim_2,dim_3,dim_4,dim_5,dim_6
0,1,0.046246,0.098058,0.083482,0.234401,0.072280,0.402023,0.060046
1,2,0.094474,0.170529,-0.099967,0.240552,-0.129931,0.214152,0.134355
2,3,0.068170,-0.198229,0.085211,-0.028215,-0.206034,0.202491,0.380080
3,4,0.069248,0.005351,-0.126389,0.074981,-0.229945,0.245645,0.174395
4,5,0.047956,0.110874,-0.167514,0.127849,-0.314073,0.280456,0.161102


In [16]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

if "id" not in resenas.columns:
    raise ValueError(
        "El archivo de reseñas no contiene la columna id."
    )

if resenas["id"].duplicated().any():
    raise ValueError(
        "Hay IDs duplicados en los metadatos."
    )

categorias = [
    columna
    for columna in ["genero", "valoracion", "periodo"]
    if columna in resenas.columns
]

if not categorias:
    raise ValueError(
        "No se encontraron las columnas de categorías."
    )

documentos_meta = documentos[["id"]].merge(
    resenas[["id"] + categorias],
    on="id",
    how="left",
    validate="one_to_one"
)

assert len(documentos_meta) == len(X_cbow)
assert len(documentos_meta) == len(X_skipgram)

def calcular_similitud_categorias(
    matriz, etiquetas
):
    etiquetas = etiquetas.fillna(
        "Sin dato"
    ).astype(str)

    grupos = sorted(etiquetas.unique())

    if len(grupos) < 2:
        return None

    centro_global = matriz.mean(axis=0)

    centroides = np.asarray([
        matriz[
            etiquetas.to_numpy() == grupo
        ].mean(axis=0)
        for grupo in grupos
    ])

    if len(grupos) > 2:
        vectores_comparacion = (
            centroides - centro_global
        )
        metodo = "Centrado global"
    else:
        vectores_comparacion = centroides
        metodo = "Sin centrar (2 categorías)"

    similitudes = cosine_similarity(
        vectores_comparacion
    )

    tabla = pd.DataFrame(
        similitudes,
        index=grupos,
        columns=grupos
    )

    return tabla, metodo

for categoria in categorias:

    for nombre, matriz in [
        ("CBOW", X_cbow),
        ("Skip-Gram", X_skipgram)
    ]:

        resultado = calcular_similitud_categorias(
            matriz,
            documentos_meta[categoria]
        )

        if resultado is None:
            print(
                f"{categoria} - {nombre}: "
                "No hay suficientes categorías."
            )
            continue

        tabla, metodo = resultado

        print(
            f"\nSIMILITUD: {categoria.upper()} - {nombre}"
        )
        print("Método:", metodo)

        display(tabla.round(4))

        tabla.to_csv(
            SALIDA /
            f"similitud_{categoria}_{nombre.lower()}.csv",
            encoding="utf-8-sig"
        )

print("\nANÁLISIS DE SIMILITUD FINALIZADO")



SIMILITUD: GENERO - CBOW
Método: Centrado global


,Action,Animation,Comedy,Drama,Horror
Action,1.0000,-0.3507,-0.4027,-0.3919,0.2178
Animation,-0.3507,1.0000,0.1608,-0.2202,-0.5420
Comedy,-0.4027,0.1608,1.0000,-0.0347,-0.4928
Drama,-0.3919,-0.2202,-0.0347,1.0000,-0.3939
Horror,0.2178,-0.5420,-0.4928,-0.3939,1.0000



SIMILITUD: GENERO - Skip-Gram
Método: Centrado global


,Action,Animation,Comedy,Drama,Horror
Action,1.0000,-0.4134,-0.3390,-0.2799,0.1902
Animation,-0.4134,1.0000,0.0938,-0.1053,-0.6307
Comedy,-0.3390,0.0938,1.0000,-0.1244,-0.4989
Drama,-0.2799,-0.1053,-0.1244,1.0000,-0.3144
Horror,0.1902,-0.6307,-0.4989,-0.3144,1.0000



SIMILITUD: VALORACION - CBOW
Método: Centrado global


,intermedia,negativa,positiva
intermedia,1.0000,0.3885,-0.6507
negativa,0.3885,1.0000,-0.9525
positiva,-0.6507,-0.9525,1.0000


... (3 outputs left out)

,2010_en_adelante,antes_2010
2010_en_adelante,1.0000,0.9986
antes_2010,0.9986,1.0000



SIMILITUD: PERIODO - Skip-Gram
Método: Sin centrar (2 categorías)


,2010_en_adelante,antes_2010
2010_en_adelante,1.0000,0.9994
antes_2010,0.9994,1.0000



ANÁLISIS DE SIMILITUD FINALIZADO


In [17]:
import pandas as pd
from collections import Counter

documentos_categorias = documentos[
    ["id", "tokens"]
].merge(
    resenas[["id"] + categorias],
    on="id",
    how="left",
    validate="one_to_one"
)

def vocabulario_exclusivo(df, columna):
    etiquetas = df[columna].fillna("Sin dato").astype(str)
    grupos = sorted(etiquetas.unique())

    vocabularios = {}
    frecuencias = {}

    for grupo in grupos:
        tokens_grupo = [
            palabra
            for lista in df.loc[etiquetas == grupo, "tokens"]
            for palabra in lista
        ]

        frecuencias[grupo] = Counter(tokens_grupo)
        vocabularios[grupo] = set(tokens_grupo)

    resultados = []

    for grupo in grupos:
        vocabulario_otros = set().union(
            *[
                vocabularios[otro]
                for otro in grupos
                if otro != grupo
            ]
        )

        exclusivos = (
            vocabularios[grupo] - vocabulario_otros
        )

        mas_frecuentes = sorted(
            exclusivos,
            key=lambda palabra: (
                -frecuencias[grupo][palabra],
                palabra
            )
        )

        resultados.append({
            "categoria": grupo,
            "cantidad_exclusivas": len(exclusivos),
            "top_20_exclusivas": ", ".join(
                mas_frecuentes[:20]
            )
        })

    return pd.DataFrame(resultados)


for categoria in categorias:
    print(f"\nVOCABULARIO EXCLUSIVO: {categoria.upper()}")

    tabla = vocabulario_exclusivo(
        documentos_categorias,
        categoria
    )

    display(tabla)

    tabla.to_csv(
        SALIDA / f"vocabulario_exclusivo_{categoria}.csv",
        index=False,
        encoding="utf-8-sig"
    )

print("\nVOCABULARIO EXCLUSIVO FINALIZADO")



VOCABULARIO EXCLUSIVO: GENERO


,categoria,cantidad_exclusivas,top_20_exclusivas
0,Action,2835,"helicópteros, mayas, cruzada, mosqueteros, rod..."
1,Animation,2567,"melocotón, pollos, ogro, animadores, arcilla, ..."
2,Comedy,2645,"bodas, resaca, burdo, patos, elfo, monjas, rau..."
3,Drama,3193,"iraní, infectados, israelíes, abogados, amargu..."
4,Horror,2634,"scream, motosierra, urbanas, serbia, autopsia,..."



VOCABULARIO EXCLUSIVO: VALORACION


,categoria,cantidad_exclusivas,top_20_exclusivas
0,intermedia,2062,"chanticleer, entomóloga, abortar, anticlimátic..."
1,negativa,3057,"evite, oxígeno, humeante, duques, escribiera, ..."
2,positiva,10820,"desgarrador, subestimada, empleado, escéptico,..."



VOCABULARIO EXCLUSIVO: PERIODO


,categoria,cantidad_exclusivas,top_20_exclusivas
0,2010_en_adelante,8510,"dir, tropos, serbia, prácticos, autopsia, pers..."
1,antes_2010,7188,"banal, consiga, digas, colgado, comentado, des..."



VOCABULARIO EXCLUSIVO FINALIZADO


In [18]:
import pandas as pd
from IPython.display import display

print("=" * 60)
print("RESUMEN DEL ANÁLISIS NLP ")
print("=" * 60)

print("\n1. INFORMACIÓN GENERAL")
print("Reseñas procesadas:", len(documentos))
print("Oraciones procesadas:", len(oraciones))
print("Vocabulario CBOW:", len(modelo_cbow.wv))
print("Vocabulario Skip-Gram:", len(modelo_skipgram.wv))
print("Dimensiones Word2Vec:", modelo_cbow.vector_size)

print("\n2. TOP 10 - BAG OF WORDS")
display(tabla_bow.head(10))

print("\n3. TOP 10 - TF-IDF PROMEDIO")
display(tabla_tfidf.head(10))

print("\n4. EJEMPLOS DE SIMILITUD SEMÁNTICA")
display(tabla_semantica.head(15))

print("\n5. RESULTADOS DE ANALOGÍAS")
display(tabla_analogias)

print("\nACIERTOS POR MODELO")
display(
    tabla_analogias.groupby("modelo")[
        "acierto_top5"
    ].agg(["sum", "count"])
)

print("\n6. EMBEDDINGS GENERADOS")
print("CBOW:", X_cbow.shape)
print("Skip-Gram:", X_skipgram.shape)

print("\n7. VOCABULARIO EXCLUSIVO")

for categoria in categorias:
    archivo = (
        SALIDA /
        f"vocabulario_exclusivo_{categoria}.csv"
    )

    if archivo.exists():
        print("\nCategoría:", categoria)
        display(pd.read_csv(archivo))

print("\n8. ARCHIVOS GENERADOS")

for archivo in sorted(SALIDA.iterdir()):
    if archivo.is_file():
        print(archivo.name)

print("\nRESUMEN FINALIZADO")


RESUMEN DEL ANÁLISIS NLP 

1. INFORMACIÓN GENERAL
Reseñas procesadas: 6283
Oraciones procesadas: 61360
Vocabulario CBOW: 13232
Vocabulario Skip-Gram: 13232
Dimensiones Word2Vec: 100

2. TOP 10 - BAG OF WORDS


,palabra,frecuencia
21122,película,21120
19760,no,15698
19441,más,6787
21124,películas,3984
19421,muy,3945
14692,historia,3755
29104,ver,3516
27452,tan,2966
27868,tiene,2948
23576,realmente,2829



3. TOP 10 - TF-IDF PROMEDIO


,palabra,tfidf_promedio
21122,película,0.077549
19760,no,0.060102
19441,más,0.033071
19421,muy,0.025333
21124,películas,0.025284
14692,historia,0.024111
29104,ver,0.023424
27452,tan,0.020666
23576,realmente,0.020432
27868,tiene,0.019930



4. EJEMPLOS DE SIMILITUD SEMÁNTICA


,modelo,campo,palabra,termino_usado,similar,similitud
0,CBOW,Cine,película,película,experiencia,0.6890
1,CBOW,Cine,película,película,característica,0.6593
2,CBOW,Cine,película,película,secuela,0.6493
3,CBOW,Cine,película,película,comedia,0.6353
4,CBOW,Cine,película,película,sorpresa,0.6050
5,CBOW,Cine,actor,actor,papel,0.8887
6,CBOW,Cine,actor,actor,actriz,0.8753
7,CBOW,Cine,actor,actor,villano,0.7980
8,CBOW,Cine,actor,actor,casting,0.7931
9,CBOW,Cine,actor,actor,interpretar,0.7487


... (7 outputs left out)

,categoria,cantidad_exclusivas,top_20_exclusivas
0,intermedia,2062,"chanticleer, entomóloga, abortar, anticlimátic..."
1,negativa,3057,"evite, oxígeno, humeante, duques, escribiera, ..."
2,positiva,10820,"desgarrador, subestimada, empleado, escéptico,..."



Categoría: periodo


,categoria,cantidad_exclusivas,top_20_exclusivas
0,2010_en_adelante,8510,"dir, tropos, serbia, prácticos, autopsia, pers..."
1,antes_2010,7188,"banal, consiga, digas, colgado, comentado, des..."



8. ARCHIVOS GENERADOS
analogias_word2vec.csv
bow.npz
campos_semanticos.csv
frecuencias_bow.csv
ids_resenas.csv
importancia_tfidf.csv
similitud_genero_cbow.csv
similitud_genero_skip-gram.csv
similitud_periodo_cbow.csv
similitud_periodo_skip-gram.csv
similitud_valoracion_cbow.csv
similitud_valoracion_skip-gram.csv
tfidf.npz
vectores_cbow.csv
vectores_skipgram.csv
vocabulario_exclusivo_genero.csv
vocabulario_exclusivo_periodo.csv
vocabulario_exclusivo_valoracion.csv
word2vec_cbow.model
word2vec_skipgram.model

RESUMEN FINALIZADO


## Interpretación de los resultados observados

Los siguientes valores proceden de la **ejecución del notebook aportado**, y pueden variar al reentrenar Word2Vec en otro entorno.

- **Corpus y vectores:** se procesaron **6.283 reseñas** y **61.360 oraciones**. BoW y TF-IDF generaron matrices de **6.283 × 29.862**; ambos modelos Word2Vec conservaron **13.232 términos** y **100 dimensiones**. Los embeddings por reseña tienen **6.283 × 100**, con IDs alineados.
- **BoW / TF-IDF:** «película» aparece **21.120** veces y alcanza el mayor TF-IDF promedio observado (**0,077549**); «no» aparece **15.698** veces y tiene un promedio de **0,060102**. Estas cifras describen frecuencia e importancia media en el corpus, no relevancia universal de las palabras.
- **Campos semánticos:** para «película», CBOW recuperó «experiencia» (**0,6890**), «característica» (**0,6593**) y «secuela» (**0,6493**). La proximidad coseno sugiere asociaciones contextuales; no garantiza sinonimia.
- **Analogías:** CBOW acertó **1 de 6** y Skip-Gram **1 de 6** en el top 5. El resultado muestra que las relaciones vectoriales propuestas son difíciles para estos modelos y este corpus; no debe presentarse como una precisión general del modelo.
- **Géneros:** con centroides centrados por la media global de reseñas, CBOW obtuvo **0,2178** entre *Action* y *Horror* y **−0,5420** entre *Animation* y *Horror*. El centrado cambia la interpretación: se comparan desviaciones respecto del corpus global.
- **Valoración:** los centroides de *negativa* y *positiva* fueron muy opuestos tras el centrado (**−0,9525** CBOW; **−0,9494** Skip-Gram). No equivale a una medida de exactitud de clasificación.
- **Período:** al existir solo dos categorías, se compararon centroides **sin centrar** para evitar una anticorrelación forzada. La similitud fue **0,9986** (CBOW) y **0,9994** (Skip-Gram), lo que indica gran proximidad en las direcciones medias, no ausencia de diferencias temporales.
- **Vocabulario exclusivo:** *Drama* reunió **3.193** palabras exclusivas entre géneros; la valoración *positiva*, **10.820**. Las cantidades dependen del tamaño de cada grupo y de la presencia de palabras poco frecuentes, por lo que **no** prueban por sí solas una mayor diversidad léxica normalizada.

**Limitaciones:** el filtrado por POS incluye algunas palabras funcionales clasificadas como contenido; los embeddings son promedios simples sin ponderación TF-IDF; las seis analogías son exploratorias; la comparación de centroides puede verse afectada por el desequilibrio de clases. Para nuevas ejecuciones, revisar el aviso de Gensim si aparecen errores adicionales.


In [19]:
from pathlib import Path

print("=" * 55)
print("Archivos del proyecto")
print("=" * 55)

for carpeta in ["data", "src", "notebooks"]:
    ruta = RAIZ / carpeta
    print(f"{carpeta}: {'OK' if ruta.is_dir() else 'FALTA'}")

entregables = [
    RAIZ / "src" / "embeddings_w2v.py",
    RAIZ / "notebooks" / "03_word2vec_analisis.ipynb",
    RAIZ / "README.md"
]

print("\nENTREGABLES:")

for archivo in entregables:
    estado = "EXISTE" if archivo.is_file() else "FALTA"
    print(f"{archivo.relative_to(RAIZ)}: {estado}")

print("\nRESULTADOS EXPORTADOS:")

archivos = [
    "bow.npz",
    "tfidf.npz",
    "ids_resenas.csv",
    "word2vec_cbow.model",
    "word2vec_skipgram.model",
    "campos_semanticos.csv",
    "analogias_word2vec.csv",
    "vectores_cbow.csv",
    "vectores_skipgram.csv"
]

for nombre in archivos:
    ruta = SALIDA / nombre
    print(f"{nombre}: {'OK' if ruta.is_file() else 'FALTA'}")

print("\nVerificación completada")


Archivos del proyecto
data: OK
src: OK
notebooks: OK

ENTREGABLES:
src\embeddings_w2v.py: EXISTE
notebooks\03_word2vec_analisis.ipynb: FALTA
README.md: EXISTE

RESULTADOS EXPORTADOS:
bow.npz: OK
tfidf.npz: OK
ids_resenas.csv: OK
word2vec_cbow.model: OK
word2vec_skipgram.model: OK
campos_semanticos.csv: OK
analogias_word2vec.csv: OK
vectores_cbow.csv: OK
vectores_skipgram.csv: OK

Verificación completada
